# Power-from-Shore Feasibility Analysis

Uses the NeqSim load list as starting point for a **power-from-shore (PFS)**
electrification study. After the process simulation gives us the platform
electrical demand, we add engineering calculations for:

1. **Submarine cable sizing** (HVAC / HVDC) — conductor cross-section, losses, reactive power
2. **Onshore & offshore substation sizing** — transformers, switchgear
3. **Cost estimation** — cable CAPEX, substation CAPEX, installation, OPEX
4. **CO₂ emissions comparison** — gas turbine vs power-from-shore
5. **Economics** — NPV of electrification including Norwegian CO₂ tax
6. **Regional comparison** — Norway, UK, Brazil, Gulf of Mexico

### Standards & References
- IEC 63026 — Submarine power cables
- CIGRÉ TB 610 — Offshore generation cable connections
- DNV-RP-J301 — Subsea power cables in shallow water
- IEC 60287 — Current rating of cables
- Norwegian government: Electrification of the NCS (Energy White Paper)

In [ ]:
# Setup
import subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    if ns is not None:
        ns = neqsim_classes(ns)
        NEQSIM_MODE = "devtools"
        print("NeqSim loaded via devtools")
    else:
        raise RuntimeError("devtools returned None")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip")

import json
import jpype
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
Compressor = jneqsim.process.equipment.compressor.Compressor
Pump = jneqsim.process.equipment.pump.Pump
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Separator = jneqsim.process.equipment.separator.Separator
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

# NeqSim environmental reporter
EnvironmentalReporter = jpype.JClass(
    "neqsim.process.fielddevelopment.evaluation.EnvironmentalReporter")
PowerSupplyType = jpype.JClass(
    "neqsim.process.fielddevelopment.evaluation.EnvironmentalReporter$PowerSupplyType")

print(f"Mode: {NEQSIM_MODE}")

## 1. Process Simulation — Determine Platform Demand

Build a representative offshore gas-condensate processing platform and
extract the electrical load list.

In [ ]:
# Create well fluid (gas-condensate)
fluid = SystemSrkEos(273.15 + 80.0, 60.0)
fluid.addComponent("methane", 0.70)
fluid.addComponent("ethane", 0.10)
fluid.addComponent("propane", 0.05)
fluid.addComponent("i-butane", 0.02)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("n-pentane", 0.01)
fluid.addComponent("n-hexane", 0.01)
fluid.addComponent("nitrogen", 0.03)
fluid.addComponent("CO2", 0.04)
fluid.addComponent("water", 0.02)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

# Build process
process = ProcessSystem()

feed = Stream("Well Stream", fluid)
feed.setFlowRate(80000.0, "kg/hr")
process.add(feed)

hp_sep = Separator("HP Separator", feed)
process.add(hp_sep)

gas_comp1 = Compressor("Gas Compressor", hp_sep.getGasOutStream())
gas_comp1.setOutletPressure(120.0)
process.add(gas_comp1)

gas_cooler = Cooler("Gas Aftercooler", gas_comp1.getOutletStream())
gas_cooler.setOutTemperature(273.15 + 40.0)
process.add(gas_cooler)

gas_comp2 = Compressor("Booster Compressor", gas_cooler.getOutletStream())
gas_comp2.setOutletPressure(180.0)
process.add(gas_comp2)

export_pump = Pump("Export Pump", hp_sep.getLiquidOutStream())
export_pump.setOutletPressure(80.0)
process.add(export_pump)

process.run()
process.runAllElectricalDesigns()
load_list = process.getElectricalLoadList()

platform_demand_kw = float(load_list.getMaximumDemandKW())
platform_demand_kva = float(load_list.getMaximumDemandKVA())
platform_pf = float(load_list.getOverallPowerFactor())

print("=== Platform Electrical Demand ===")
print(f"Maximum demand:       {platform_demand_kw:.0f} kW / {platform_demand_kva:.0f} kVA")
print(f"Power factor:         {platform_pf:.3f}")
print(f"Equipment count:      {load_list.getLoadCount()}")

## 2. Submarine Cable Sizing (HVAC)

The submarine cable must deliver the platform demand over a long distance.
Key engineering calculations:

### Active power capacity

$$S_{\text{cable}} = \sqrt{3} \cdot V_{\text{rated}} \cdot I_{\text{rated}}$$

### Cable resistive losses (3-phase AC)

$$P_{\text{loss}} = 3 \cdot I^2 \cdot R_{\text{AC}} \cdot L \quad \text{where} \quad R_{\text{AC}} = \frac{\rho}{A} (1 + y_s + y_p)$$

- $\rho$ = conductor resistivity (Cu: $1.72 \times 10^{-8}$ Ω·m at 20°C)
- $y_s$ = skin effect factor (~0.02–0.08)
- $y_p$ = proximity effect factor (~0.01–0.04)

### Dielectric losses (per IEC 60287)

$$W_d = 2\pi f \cdot C \cdot U_0^2 \cdot \tan\delta \cdot L$$

### Charging current (reactive power limit for HVAC)

$$I_c = 2\pi f \cdot C \cdot U_0 \quad \Rightarrow \quad Q_c = 3 \cdot U_0 \cdot I_c \cdot L$$

For long HVAC cables, charging current consumes cable capacity. Beyond
~80–100 km at 132 kV, **HVDC becomes necessary**.

In [ ]:
def size_submarine_cable(demand_mw, distance_km, voltage_kv=132.0,
                         conductor="copper", temperature_c=90.0):
    """Size an HVAC submarine 3-core XLPE cable per IEC 60287 / IEC 63026.

    Returns a dict with cable data, losses, and reactive power.
    """
    # Conductor material properties
    if conductor == "copper":
        rho_20 = 1.7241e-8   # ohm.m at 20 C
        alpha = 3.93e-3      # temperature coefficient
        density = 8900       # kg/m3
    else:  # aluminium
        rho_20 = 2.8264e-8
        alpha = 4.03e-3
        density = 2700

    # Temperature-corrected resistivity
    rho = rho_20 * (1 + alpha * (temperature_c - 20.0))

    # Standard cable sizes (mm2) and ratings (A) for 132 kV XLPE submarine
    # Source: typical manufacturer data (Nexans, Prysmian, NKT)
    cable_table = [
        # (mm2, continuous A at 90C, approx. weight kg/m, outer diam mm)
        (185,  440, 48, 210),
        (240,  510, 52, 220),
        (300,  575, 56, 230),
        (400,  650, 62, 242),
        (500,  730, 68, 255),
        (630,  820, 78, 270),
        (800,  920, 90, 290),
        (1000, 1030, 105, 310),
        (1200, 1120, 118, 325),
        (1600, 1260, 140, 350),
        (2000, 1380, 165, 375),
    ]

    # Phase voltage
    U0 = voltage_kv / np.sqrt(3)  # kV phase-to-ground

    # Required current
    S_required = demand_mw / (np.sqrt(3) * voltage_kv * 1e-3)  # kA -> use MVA / (sqrt3*kV)
    I_required = demand_mw * 1e3 / (np.sqrt(3) * voltage_kv)   # A

    # Select cable
    selected = None
    for mm2, amp, wt, od in cable_table:
        if amp >= I_required * 1.15:  # 15% margin
            selected = (mm2, amp, wt, od)
            break
    if selected is None:
        selected = cable_table[-1]  # largest available

    mm2, I_rated, cable_weight_per_m, cable_od_mm = selected
    A = mm2 * 1e-6  # m2

    # AC resistance per km (including skin & proximity effect)
    R_dc_per_km = rho / A * 1000  # ohm/km
    y_s = 0.04  # skin effect factor
    y_p = 0.02  # proximity effect factor
    R_ac_per_km = R_dc_per_km * (1 + y_s + y_p)  # ohm/km

    # Resistive losses
    I_load = I_required
    P_loss_mw = 3 * I_load**2 * R_ac_per_km * distance_km / 1e6  # MW

    # Capacitance (typical for 132 kV XLPE submarine: ~0.15-0.25 uF/km)
    C_per_km = 0.20e-6  # F/km (typical)
    f = 50  # Hz

    # Charging current per phase
    I_c = 2 * np.pi * f * C_per_km * distance_km * U0 * 1e3  # A
    Q_charging_mvar = 3 * (U0 * 1e3) * I_c / 1e6  # Mvar

    # Dielectric losses
    tan_delta = 0.001  # XLPE
    W_d_mw = 2 * np.pi * f * C_per_km * distance_km * (U0 * 1e3)**2 * tan_delta * 3 / 1e6

    # Sheath losses (approximate as 5% of resistive)
    P_sheath_mw = P_loss_mw * 0.05

    # Total cable losses
    P_total_loss = P_loss_mw + W_d_mw + P_sheath_mw
    loss_percent = P_total_loss / demand_mw * 100 if demand_mw > 0 else 0

    # Remaining capacity after charging current
    I_available = np.sqrt(max(I_rated**2 - I_c**2, 0))
    P_available_mw = np.sqrt(3) * voltage_kv * I_available / 1e3

    # Voltage drop
    X_per_km = 0.12  # ohm/km (typical 132 kV submarine)
    pf = 0.85
    V_drop_pct = (np.sqrt(3) * I_load * distance_km *
                  (R_ac_per_km * pf + X_per_km * np.sqrt(1 - pf**2))) / (voltage_kv * 1e3) * 100

    # Weight
    cable_total_weight_t = cable_weight_per_m * distance_km * 1000 / 1000  # tonnes

    # Technology recommendation
    if distance_km > 100 and voltage_kv <= 132:
        tech_recommendation = "HVDC recommended (>100 km)"
    elif distance_km > 70:
        tech_recommendation = "HVAC marginal — consider HVDC or mid-point compensation"
    else:
        tech_recommendation = "HVAC feasible"

    return {
        "demand_mw": demand_mw,
        "distance_km": distance_km,
        "voltage_kv": voltage_kv,
        "conductor_mm2": mm2,
        "rated_current_a": I_rated,
        "load_current_a": round(I_load, 1),
        "charging_current_a": round(I_c, 1),
        "available_capacity_mw": round(P_available_mw, 1),
        "cable_od_mm": cable_od_mm,
        "cable_weight_kg_m": cable_weight_per_m,
        "cable_total_weight_t": round(cable_total_weight_t, 0),
        "R_ac_ohm_km": round(R_ac_per_km, 4),
        "resistive_loss_mw": round(P_loss_mw, 2),
        "dielectric_loss_mw": round(W_d_mw, 3),
        "sheath_loss_mw": round(P_sheath_mw, 3),
        "total_loss_mw": round(P_total_loss, 2),
        "loss_percent": round(loss_percent, 1),
        "charging_mvar": round(Q_charging_mvar, 1),
        "voltage_drop_pct": round(V_drop_pct, 1),
        "technology": tech_recommendation,
    }


# Size cable for our platform
demand_mw = platform_demand_kw / 1000.0
distance_km = 150.0  # typical NCS distance

cable = size_submarine_cable(demand_mw, distance_km, voltage_kv=132.0)

print("=== Submarine Cable Design ===")
for k, v in cable.items():
    print(f"  {k:30s}: {v}")

## 3. Substation Sizing & Equipment List

A PFS system requires:

| Equipment | Location | Typical Rating |
|-----------|----------|----------------|
| Step-up transformer | Onshore | MV → HV (e.g. 22 → 132 kV) |
| GIS switchgear | Onshore | 132 kV, SF₆-free option |
| Reactive compensation | Onshore + mid-point | Shunt reactors |
| Submarine cable | Seabed | 3-core XLPE |
| Step-down transformer | Offshore | HV → MV (132 → 11 kV) |
| MV switchgear | Offshore | 11 kV GIS |
| VFDs / distribution | Offshore | Per equipment |

In [ ]:
def estimate_pfs_cost(demand_mw, distance_km, voltage_kv=132.0,
                      water_depth_m=300.0, n_cables=1, region="norway"):
    """Estimate complete PFS system CAPEX and OPEX.

    Cost basis: Industry benchmarks for NCS electrification projects
    (Utsira High, Troll, Johan Sverdrup).
    """
    # ------------------------------------------------------------------
    # Cable cost (MNOK/km) — depends on voltage, conductor size, type
    # Typical 132 kV 3-core XLPE submarine: 8-20 MNOK/km
    # ------------------------------------------------------------------
    cable_info = size_submarine_cable(demand_mw, distance_km, voltage_kv)
    mm2 = cable_info["conductor_mm2"]

    # Cable supply cost (MNOK/km), scales with cross-section
    cable_supply_cost_per_km = 4.0 + mm2 * 0.012  # rough fit to market data

    # Installation cost (MNOK/km), depends on water depth
    if water_depth_m < 100:
        install_factor = 3.5
    elif water_depth_m < 300:
        install_factor = 5.0
    elif water_depth_m < 500:
        install_factor = 7.5
    else:
        install_factor = 10.0

    cable_capex = (cable_supply_cost_per_km + install_factor) * distance_km * n_cables

    # ------------------------------------------------------------------
    # Onshore substation (MNOK): transformer, GIS, reactive compensation
    # ------------------------------------------------------------------
    onshore_trafo = demand_mw * 0.8  # ~0.8 MNOK/MW for HV transformer
    onshore_gis = 30.0 + demand_mw * 0.3  # GIS switchgear
    onshore_reactive = cable_info["charging_mvar"] * 0.5  # 0.5 MNOK/Mvar for reactors
    onshore_building = 25.0  # building/civil
    onshore_grid_connection = 20.0 + demand_mw * 0.2  # grid reinforcement
    onshore_capex = onshore_trafo + onshore_gis + onshore_reactive + onshore_building + onshore_grid_connection

    # ------------------------------------------------------------------
    # Offshore receiving station (MNOK): transformer, GIS, module
    # ------------------------------------------------------------------
    offshore_trafo = demand_mw * 1.2  # higher cost offshore
    offshore_gis = 20.0 + demand_mw * 0.5  # compact GIS
    offshore_module = 40.0 + demand_mw * 1.5  # module/platform modification
    offshore_capex = offshore_trafo + offshore_gis + offshore_module

    # ------------------------------------------------------------------
    # Engineering, procurement, project management (~15%)
    # ------------------------------------------------------------------
    subtotal = cable_capex + onshore_capex + offshore_capex
    epc_management = subtotal * 0.15

    total_capex = subtotal + epc_management

    # Regional multiplier
    region_multipliers = {
        "norway": 1.0,
        "uk": 0.95,
        "brazil": 0.85,
        "gom": 1.10,       # Gulf of Mexico
        "west_africa": 0.90,
        "australia": 1.15,
    }
    multiplier = region_multipliers.get(region, 1.0)
    total_capex *= multiplier

    # ------------------------------------------------------------------
    # OPEX (MNOK/year): electricity cost + maintenance
    # ------------------------------------------------------------------
    electricity_prices = {  # NOK/kWh
        "norway": 0.50,
        "uk": 0.80,
        "brazil": 0.40,
        "gom": 0.60,
        "west_africa": 0.70,
        "australia": 0.90,
    }
    elec_price = electricity_prices.get(region, 0.60)
    operating_hours = 8400  # hours/year
    load_factor = 0.85

    # Include cable losses in electricity cost
    total_power_mw = demand_mw * (1 + cable_info["loss_percent"] / 100)
    electricity_opex = total_power_mw * 1000 * elec_price * operating_hours * load_factor / 1e6

    maintenance_opex = total_capex * 0.015  # 1.5% of CAPEX
    total_opex = electricity_opex + maintenance_opex

    return {
        "cable_info": cable_info,
        "cable_capex_mnok": round(cable_capex, 1),
        "onshore_substation_mnok": round(onshore_capex, 1),
        "offshore_receiving_mnok": round(offshore_capex, 1),
        "epc_management_mnok": round(epc_management, 1),
        "total_capex_mnok": round(total_capex, 1),
        "electricity_opex_mnok_yr": round(electricity_opex, 1),
        "maintenance_opex_mnok_yr": round(maintenance_opex, 1),
        "total_opex_mnok_yr": round(total_opex, 1),
        "region": region,
        "electricity_price_nok_kwh": elec_price,
        "equipment_list": {
            "onshore_transformer_mva": round(demand_mw / 0.85 * 1.15, 0),
            "onshore_gis_kv": voltage_kv,
            "shunt_reactors_mvar": round(cable_info["charging_mvar"] / 2, 1),
            "submarine_cable_mm2": mm2,
            "submarine_cable_km": distance_km,
            "n_cables": n_cables,
            "offshore_transformer_mva": round(demand_mw / 0.85 * 1.15, 0),
            "offshore_gis_kv": voltage_kv,
        },
    }


# Estimate for our platform
pfs = estimate_pfs_cost(demand_mw, distance_km=150.0, voltage_kv=132.0,
                        water_depth_m=300.0, region="norway")

print("=== PFS Cost Estimate (Norway, 150 km, 132 kV) ===")
print(f"  Cable CAPEX:           {pfs['cable_capex_mnok']:>8.0f} MNOK")
print(f"  Onshore substation:    {pfs['onshore_substation_mnok']:>8.0f} MNOK")
print(f"  Offshore receiving:    {pfs['offshore_receiving_mnok']:>8.0f} MNOK")
print(f"  EPC / management:      {pfs['epc_management_mnok']:>8.0f} MNOK")
print(f"  Total CAPEX:           {pfs['total_capex_mnok']:>8.0f} MNOK")
print(f"  ---")
print(f"  Electricity OPEX:      {pfs['electricity_opex_mnok_yr']:>8.0f} MNOK/yr")
print(f"  Maintenance OPEX:      {pfs['maintenance_opex_mnok_yr']:>8.0f} MNOK/yr")
print(f"  Total OPEX:            {pfs['total_opex_mnok_yr']:>8.0f} MNOK/yr")

print(f"\n--- Equipment List ---")
for k, v in pfs["equipment_list"].items():
    print(f"  {k:30s}: {v}")

## 4. CO₂ Emissions — Gas Turbine vs Power-from-Shore

Use NeqSim's `EnvironmentalReporter` to compare emissions under the
two power supply scenarios.

| Supply | CO₂ factor (kg/kWh) | Source |
|--------|----------------------|--------|
| Gas turbine | 0.55 | Offshore simple-cycle |
| Power from shore (Norway) | 0.02 | Nordic hydro grid |
| Power from shore (UK) | 0.20 | UK grid mix |
| Combined cycle | 0.40 | Onshore CCGT |

In [ ]:
# CO2 comparison using NeqSim EnvironmentalReporter
operating_hours = 8400  # hours/year

reporter_gt = EnvironmentalReporter(PowerSupplyType.GAS_TURBINE)
reporter_gt.setOperatingHours(float(operating_hours))
co2_gt = float(reporter_gt.getCO2FromPower(float(platform_demand_kw)))

reporter_pfs = EnvironmentalReporter(PowerSupplyType.POWER_FROM_SHORE)
reporter_pfs.setOperatingHours(float(operating_hours))
co2_pfs = float(reporter_pfs.getCO2FromPower(float(platform_demand_kw)))

co2_reduction = co2_gt - co2_pfs
co2_reduction_pct = co2_reduction / co2_gt * 100 if co2_gt > 0 else 0

# CO2 tax (Norwegian)  ~1000 NOK/tonne in 2025, rising to ~2000 by 2030
co2_tax_2025 = 1000  # NOK/tonne
co2_tax_2030 = 2000  # NOK/tonne
co2_tax_saved_2025 = co2_reduction * co2_tax_2025 / 1e6  # MNOK/yr
co2_tax_saved_2030 = co2_reduction * co2_tax_2030 / 1e6  # MNOK/yr

print("=== CO\u2082 Emissions Comparison ===")
print(f"  Gas turbine:           {co2_gt:>10,.0f} tonnes/year")
print(f"  Power from shore:      {co2_pfs:>10,.0f} tonnes/year")
print(f"  Annual reduction:      {co2_reduction:>10,.0f} tonnes/year ({co2_reduction_pct:.0f}%)")
print(f"")
print(f"  CO\u2082 tax saved (2025):  {co2_tax_saved_2025:>10.1f} MNOK/yr  (@{co2_tax_2025} NOK/t)")
print(f"  CO\u2082 tax saved (2030):  {co2_tax_saved_2030:>10.1f} MNOK/yr  (@{co2_tax_2030} NOK/t)")

In [ ]:
# Visualize emissions comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Annual CO2 bar chart
ax = axes[0]
options = ["Gas Turbine", "Power\nfrom Shore"]
co2_vals = [co2_gt, co2_pfs]
bars = ax.bar(options, co2_vals, color=["#e15759", "#59a14f"], edgecolor="white", width=0.5)
ax.set_ylabel("CO\u2082 Emissions (tonnes/year)")
ax.set_title("Annual CO\u2082 Emissions")
for bar, val in zip(bars, co2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + max(co2_vals)*0.02,
            f"{val:,.0f} t", ha="center", fontweight="bold")
ax.grid(True, alpha=0.3, axis="y")

# 2. CO2 tax savings over time
ax = axes[1]
years = np.arange(2025, 2041)
co2_taxes = np.linspace(co2_tax_2025, 2500, len(years))  # rising tax trajectory
annual_savings = co2_reduction * co2_taxes / 1e6
cumulative_savings = np.cumsum(annual_savings)

ax.bar(years, annual_savings, color="steelblue", alpha=0.7, label="Annual saving")
ax2 = ax.twinx()
ax2.plot(years, cumulative_savings, "r-o", lw=2, markersize=4, label="Cumulative")
ax2.set_ylabel("Cumulative Savings (MNOK)", color="red")

ax.set_xlabel("Year")
ax.set_ylabel("Annual CO\u2082 Tax Saving (MNOK)")
ax.set_title("CO\u2082 Tax Savings (Rising Tax)")
ax.legend(loc="upper left", fontsize=9)
ax2.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.3)

# 3. CAPEX breakdown
ax = axes[2]
capex_items = ["Cable", "Onshore\nSubstation", "Offshore\nReceiving", "EPC"]
capex_vals = [pfs["cable_capex_mnok"], pfs["onshore_substation_mnok"],
              pfs["offshore_receiving_mnok"], pfs["epc_management_mnok"]]
colors_capex = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2"]
ax.bar(capex_items, capex_vals, color=colors_capex, edgecolor="white")
for i, v in enumerate(capex_vals):
    ax.text(i, v + max(capex_vals)*0.02, f"{v:.0f}", ha="center", fontweight="bold")
ax.set_ylabel("CAPEX (MNOK)")
ax.set_title(f"PFS CAPEX Breakdown — Total: {pfs['total_capex_mnok']:.0f} MNOK")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## 5. Distance & Voltage Sensitivity

How do cable losses and costs change with distance? At what point does
HVDC become necessary?

In [ ]:
# Sensitivity study: distance
distances = np.arange(20, 301, 10)
losses_pct = []
charging_mvar = []
capex_mnok = []
cable_sizes = []

for d in distances:
    c = size_submarine_cable(demand_mw, float(d), 132.0)
    p = estimate_pfs_cost(demand_mw, float(d), 132.0, 300.0, region="norway")
    losses_pct.append(c["loss_percent"])
    charging_mvar.append(c["charging_mvar"])
    capex_mnok.append(p["total_capex_mnok"])
    cable_sizes.append(c["conductor_mm2"])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cable losses vs distance
ax = axes[0, 0]
ax.plot(distances, losses_pct, "b-", lw=2)
ax.axhline(5, color="red", ls="--", alpha=0.7, label="5% loss limit")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Cable Losses (%)")
ax.set_title("Cable Losses vs Distance (132 kV HVAC)")
ax.legend()
ax.grid(True, alpha=0.3)

# Charging reactive power
ax = axes[0, 1]
ax.plot(distances, charging_mvar, "g-", lw=2)
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Charging Reactive Power (Mvar)")
ax.set_title("Reactive Power vs Distance")
ax.grid(True, alpha=0.3)

# CAPEX vs distance
ax = axes[1, 0]
ax.plot(distances, capex_mnok, "r-", lw=2)
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Total CAPEX (MNOK)")
ax.set_title("Total PFS CAPEX vs Distance")
ax.grid(True, alpha=0.3)

# Cable size vs distance
ax = axes[1, 1]
ax.step(distances, cable_sizes, "m-", lw=2, where="mid")
ax.set_xlabel("Distance (km)")
ax.set_ylabel("Conductor Size (mm\u00b2)")
ax.set_title("Required Conductor Size vs Distance")
ax.grid(True, alpha=0.3)

plt.suptitle(f"PFS Sensitivity — Platform Demand: {demand_mw:.1f} MW, 132 kV HVAC",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 6. Regional Comparison

Compare PFS feasibility across different regions with different
electricity prices, grid emission factors, and cost structures.

In [ ]:
# Regional comparison
regions = [
    # (name, distance_km, depth_m, grid_co2_kg_kwh, co2_tax_nok_t, region_key)
    ("Norway (NCS)",      150, 300, 0.02, 1000, "norway"),
    ("UK (North Sea)",    100, 100, 0.20,  200, "uk"),
    ("Brazil (pre-salt)", 250, 500, 0.08,   50, "brazil"),
    ("GoM (deepwater)",    80, 200, 0.40,   20, "gom"),
    ("W Africa",          120, 400, 0.50,   10, "west_africa"),
]

print(f"{'Region':<20} {'Dist':>5} {'CAPEX':>8} {'OPEX/yr':>8} {'CO2 GT':>8} {'CO2 PFS':>8} "
      f"{'Saved':>8} {'Tax save':>10} {'Payback':>8}")
print(f"{'':20} {'km':>5} {'MNOK':>8} {'MNOK':>8} {'t/yr':>8} {'t/yr':>8} "
      f"{'t/yr':>8} {'MNOK/yr':>10} {'years':>8}")
print("-" * 100)

region_data = []
for name, dist, depth, grid_co2, co2_tax, key in regions:
    pfs_r = estimate_pfs_cost(demand_mw, float(dist), 132.0, float(depth), region=key)

    # CO2 from gas turbine
    co2_gt_r = platform_demand_kw * 0.55 * 8400 / 1e6  # tonnes/yr
    co2_pfs_r = platform_demand_kw * grid_co2 * 8400 / 1e6
    co2_saved = co2_gt_r - co2_pfs_r
    tax_saved = co2_saved * co2_tax / 1e6  # MNOK/yr

    # Gas turbine fuel OPEX (3.0 NOK/kWh fuel equivalent for offshore GT)
    gt_fuel_opex = platform_demand_kw * 0.85 * 8400 * 2.5 / 1e6  # MNOK/yr

    # Net annual benefit of PFS vs GT
    net_annual_benefit = gt_fuel_opex - pfs_r["total_opex_mnok_yr"] + tax_saved
    payback = pfs_r["total_capex_mnok"] / net_annual_benefit if net_annual_benefit > 0 else 999

    region_data.append({
        "name": name, "capex": pfs_r["total_capex_mnok"],
        "opex": pfs_r["total_opex_mnok_yr"],
        "co2_gt": co2_gt_r, "co2_pfs": co2_pfs_r, "co2_saved": co2_saved,
        "tax_saved": tax_saved, "payback": payback,
        "gt_fuel_opex": gt_fuel_opex,
    })

    print(f"{name:<20} {dist:>5} {pfs_r['total_capex_mnok']:>8.0f} "
          f"{pfs_r['total_opex_mnok_yr']:>8.0f} {co2_gt_r:>8.0f} {co2_pfs_r:>8.0f} "
          f"{co2_saved:>8.0f} {tax_saved:>10.1f} {payback:>8.1f}")

In [ ]:
# Regional comparison plots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

names_r = [r["name"] for r in region_data]
x = np.arange(len(names_r))

# CAPEX comparison
ax = axes[0]
ax.barh(x, [r["capex"] for r in region_data], color="steelblue", edgecolor="white")
ax.set_yticks(x)
ax.set_yticklabels(names_r)
ax.set_xlabel("Total CAPEX (MNOK)")
ax.set_title("PFS CAPEX by Region")
for i, r in enumerate(region_data):
    ax.text(r["capex"] + 10, i, f"{r['capex']:.0f}", va="center", fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")

# CO2 reduction
ax = axes[1]
ax.barh(x, [r["co2_saved"] for r in region_data], color="#59a14f", edgecolor="white")
ax.set_yticks(x)
ax.set_yticklabels(names_r)
ax.set_xlabel("CO\u2082 Reduction (tonnes/year)")
ax.set_title("Annual CO\u2082 Reduction")
for i, r in enumerate(region_data):
    ax.text(r["co2_saved"] + 10, i, f"{r['co2_saved']:,.0f}", va="center", fontweight="bold")
ax.grid(True, alpha=0.3, axis="x")

# Payback period
ax = axes[2]
paybacks = [min(r["payback"], 25) for r in region_data]
colors_pb = ["green" if p < 8 else "orange" if p < 15 else "red" for p in paybacks]
ax.barh(x, paybacks, color=colors_pb, edgecolor="white")
ax.set_yticks(x)
ax.set_yticklabels(names_r)
ax.set_xlabel("Payback (years)")
ax.set_title("Simple Payback Period")
ax.axvline(8, color="gray", ls="--", alpha=0.5, label="8 yr target")
for i, r in enumerate(region_data):
    label = f"{r['payback']:.1f}" if r["payback"] < 25 else ">25"
    ax.text(min(r["payback"], 25) + 0.3, i, label, va="center", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis="x")

plt.suptitle(f"Power-from-Shore Regional Comparison — {demand_mw:.1f} MW Platform",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. HVAC vs HVDC Decision

For long distances, HVDC avoids reactive power limitations but adds
converter station costs.

In [ ]:
# HVAC vs HVDC cost crossover
distances_compare = np.arange(20, 301, 5)
hvac_costs = []
hvdc_costs = []

for d in distances_compare:
    # HVAC
    hvac = estimate_pfs_cost(demand_mw, float(d), 132.0, 300.0, region="norway")
    hvac_costs.append(hvac["total_capex_mnok"])

    # HVDC: cheaper cable per km but expensive converter stations
    converter_cost = demand_mw * 15  # ~15 MNOK/MW for converter stations (both ends)
    hvdc_cable_per_km = 6.0  # MNOK/km (2 x monopole, simpler cable)
    hvdc_install_per_km = 4.0
    hvdc_cable_total = (hvdc_cable_per_km + hvdc_install_per_km) * float(d)
    hvdc_substation_other = 50.0  # auxiliaries
    hvdc_total = (converter_cost + hvdc_cable_total + hvdc_substation_other) * 1.15  # 15% EPC
    hvdc_costs.append(hvdc_total)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(distances_compare, hvac_costs, "b-", lw=2.5, label="HVAC (132 kV)")
ax.plot(distances_compare, hvdc_costs, "r-", lw=2.5, label="HVDC")

# Find crossover
hvac_arr = np.array(hvac_costs)
hvdc_arr = np.array(hvdc_costs)
crossover_idx = np.argmin(np.abs(hvac_arr - hvdc_arr))
crossover_km = distances_compare[crossover_idx]

ax.axvline(crossover_km, color="gray", ls="--", alpha=0.7,
           label=f"Crossover: ~{crossover_km} km")
ax.fill_betweenx([min(min(hvac_costs), min(hvdc_costs)),
                  max(max(hvac_costs), max(hvdc_costs))],
                 0, crossover_km, alpha=0.05, color="blue")
ax.fill_betweenx([min(min(hvac_costs), min(hvdc_costs)),
                  max(max(hvac_costs), max(hvdc_costs))],
                 crossover_km, 300, alpha=0.05, color="red")

ax.text(crossover_km / 2, max(max(hvac_costs), max(hvdc_costs)) * 0.95,
        "HVAC\npreferred", ha="center", fontsize=11, color="blue")
ax.text((crossover_km + 300) / 2, max(max(hvac_costs), max(hvdc_costs)) * 0.95,
        "HVDC\npreferred", ha="center", fontsize=11, color="red")

ax.set_xlabel("Distance (km)")
ax.set_ylabel("Total CAPEX (MNOK)")
ax.set_title(f"HVAC vs HVDC Cost Crossover — {demand_mw:.1f} MW Platform")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"HVAC/HVDC crossover at approximately {crossover_km} km")

## Summary

This notebook demonstrated the full **power-from-shore feasibility workflow**:

1. **Process simulation** → platform electrical demand via NeqSim load list
2. **Submarine cable sizing** (IEC 60287/63026) — conductor, losses, reactive power
3. **Substation equipment** — transformers, GIS, reactive compensation
4. **Cost estimation** — cable, onshore, offshore, EPC (CAPEX + OPEX)
5. **CO₂ comparison** — gas turbine vs PFS using NeqSim `EnvironmentalReporter`
6. **Regional comparison** — Norway, UK, Brazil, GoM, West Africa
7. **HVAC vs HVDC** — cost crossover with distance

### Key Findings

- Norway NCS benefits most from PFS due to high CO₂ tax + clean grid
- HVAC is feasible up to ~80–120 km; beyond that HVDC is preferred
- Cable losses (1–5 %) are acceptable but must be included in total demand
- Reactive power compensation is critical for long HVAC cables
- The NeqSim electrical load list is the essential input for any electrification study

### Applicable Standards

| Standard | Topic |
|----------|-------|
| IEC 63026 | Submarine power cables |
| IEC 60287 | Current rating calculations |
| CIGRÉ TB 610 | Offshore cable connections |
| DNV-RP-J301 | Subsea power cables |
| NEK IEC 60502 | Power cables 1–30 kV |
| IEC 62271 | HV switchgear |

### Further Reading
- `process_plant_load_list.ipynb` — detailed load list generation
- `compressor_electrical_design.ipynb` — individual equipment electrical design
- `motor_vfd_analysis.ipynb` — motor and VFD component analysis